In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("../data/processed/lebron_model_dataset.csv")
df.head()

,GAME_DATE,MATCHUP,PTS,HOME,days_rest,pts_last3,pts_last5,pts_last10,min_last3,min_last5,fga_last3,fga_last5,fg3a_last5,fta_last5,reb_last5,ast_last5,pts_std_last5
0,2023-11-15,LAL vs. SAC,28,1,1.0,22.000000,24.0,24.3,28.666667,31.6,13.666667,15.8,4.2,5.8,7.8,4.4,7.071068
1,2023-11-17,LAL @ POR,35,0,2.0,25.333333,24.8,25.0,31.333333,31.6,14.333333,15.8,4.6,6.0,8.0,5.8,7.293833
2,2023-11-19,LAL vs. HOU,37,1,2.0,26.333333,25.8,26.4,31.000000,31.2,16.000000,15.6,5.6,6.4,8.2,7.0,8.438009
3,2023-11-21,LAL vs. UTA,17,1,2.0,33.333333,29.6,27.4,36.666667,33.8,18.666667,16.8,6.2,7.2,8.2,8.2,8.324662
4,2023-11-22,LAL vs. DAL,26,1,1.0,29.666667,26.6,27.2,33.000000,31.4,17.000000,15.4,6.4,5.4,7.4,8.8,9.813256


In [4]:
feature_columns = [
    "HOME",
    "days_rest",
    "pts_last3",
    "pts_last5",
    "pts_last10",
    "min_last3",
    "min_last5",
    "fga_last3",
    "fga_last5",
    "fg3a_last5",
    "fta_last5",
    "reb_last5",
    "ast_last5",
    "pts_std_last5",
]

target_column = "PTS"

X = df[feature_columns]
y = df[target_column]

In [5]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 48
Test size: 13


In [6]:
baseline_prediction = X_test["pts_last5"]
baseline_mae = mean_absolute_error(y_test, baseline_prediction)

print("Baseline MAE:", round(baseline_mae, 2))

Baseline MAE: 6.02


In [7]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("Linear Regression MAE:", round(mean_absolute_error(y_test, lr_pred), 2))
print("Linear Regression RMSE:", round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2))
print("Linear Regression R2:", round(r2_score(y_test, lr_pred), 2))

Linear Regression MAE: 4.98
Linear Regression RMSE: 6.99
Linear Regression R2: 0.08


In [8]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest MAE:", round(mean_absolute_error(y_test, rf_pred), 2))
print("Random Forest RMSE:", round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2))
print("Random Forest R2:", round(r2_score(y_test, rf_pred), 2))

Random Forest MAE: 5.95
Random Forest RMSE: 7.65
Random Forest R2: -0.1


In [9]:
results = pd.DataFrame({
    "Model": ["Baseline (Last 5 Avg)", "Linear Regression", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_test, baseline_prediction),
        mean_absolute_error(y_test, lr_pred),
        mean_absolute_error(y_test, rf_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, baseline_prediction)),
        np.sqrt(mean_squared_error(y_test, lr_pred)),
        np.sqrt(mean_squared_error(y_test, rf_pred))
    ],
    "R2": [
        r2_score(y_test, baseline_prediction),
        r2_score(y_test, lr_pred),
        r2_score(y_test, rf_pred)
    ]
})

results.sort_values("MAE")

,Model,MAE,RMSE,R2
1,Linear Regression,4.978886,6.993017,0.077006
2,Random Forest,5.946714,7.650027,-0.104576
0,Baseline (Last 5 Avg),6.015385,8.277681,-0.293263


In [10]:
comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Baseline": baseline_prediction.values,
    "LinearRegression": lr_pred,
    "RandomForest": rf_pred
})

comparison_df.head(10)

,Actual,Baseline,LinearRegression,RandomForest
0,40,24.6,24.837280,22.771940
1,25,27.4,20.620238,20.741653
2,20,28.6,24.479102,22.948549
3,26,26.4,18.617105,21.162085
4,23,25.8,23.085815,25.317466
5,16,26.8,31.815058,26.742431
6,40,22.0,34.521630,27.115095
7,23,25.0,20.450827,21.901185
8,25,25.6,26.038379,26.399100
9,24,25.4,28.277760,26.433650
